In [ ]:

import os
import gzip
import numpy as np

In [ ]:
def load_mnist(path, kind='train'):
    """Load MNIST data from `path`"""
    
    # Define paths to labels and images files
    labels_path = os.path.join(path, '%s-labels-idx1-ubyte.gz' % kind)
    images_path = os.path.join(path, '%s-images-idx3-ubyte.gz' % kind)

    # Load labels
    with gzip.open(labels_path, 'rb') as lbpath:
        labels = np.frombuffer(lbpath.read(), dtype=np.uint8, offset=8)

    # Load images
    with gzip.open(images_path, 'rb') as imgpath:
        images = np.frombuffer(imgpath.read(), dtype=np.uint8, offset=16).reshape(len(labels), 784)

    return images, labels

In [9]:
folder = "/".join([os.path.expanduser("~/"), "Workspace/repos/fashion-mnist/data/fashion"])
train_images, train_labels = load_mnist(folder)
len(train_images), len(train_labels)

(60000, 60000)

In [133]:
MEAN = np.array(train_images).mean()
STD = np.array(train_images).std()
MIN =  np.array(train_images).min()
MAX =  np.array(train_images).max()
MEAN, STD, MIN, MAX

(72.94035223214286, 90.02118235130519, 0, 255)

In [11]:
image = train_images[0]
np.sqrt(image.shape[0])

28.0

In [115]:
%matplotlib inline
import matplotlib.pyplot as plt
import cv2
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2

In [117]:
transforms = v2.Compose([
    # v2.RandomResizedCrop(size=(224, 224), antialias=True),
    # v2.RandomHorizontalFlip(p=0.5),
    # v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[MEAN], std=[STD]),
])

In [134]:
def preprocess(x, size_final=32):

    n = int(math.sqrt(x.shape[0]))
    x = x.reshape((n, n, -1))

    x = cv2.resize(x, (size_final, size_final), interpolation=cv2.INTER_LINEAR)
    if len(x.shape) == 2:
        x = x[..., np.newaxis]
    x = np.transpose(x, (2, 0, 1))

    x = torch.tensor(x[np.newaxis, ...], dtype=torch.float32)

    x = (x / MAX - 0.5) * 2

    return x

In [138]:
class ConvBlock(torch.nn.Module):

    def __init__(self, **kwargs):

        super().__init__()

        input_channel = kwargs.get("input_channel", 3)
        output_channel = kwargs.get("output_channel", 3)
        kernel_size = kwargs.get("kernel_size", 3)

        self.conv = nn.Conv2d(
            in_channels=input_channel,
            out_channels=output_channel,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            stride=2
        )
        # self.dropout = nn.Dropout()
        self.norm = nn.BatchNorm2d(output_channel)
        self.activate = nn.LeakyReLU()

    def forward(self, x):

        x = self.conv(x)
        # x = self.dropout(x)
        x = self.norm(x)
        x = self.activate(x)

        return x


class Discriminator(torch.nn.Module):

    def __init__(self, **kwargs):

        super().__init__()
        channels = kwargs.get("channels", [3, 5])
        channels = [
            {"input_channel": x, "output_channel": y}
            for x, y in zip([1] + channels[:-1], channels)
        ]

        self.conv_blocks = nn.Sequential(
            *[
                ConvBlock(
                    input_channel=channel_config["input_channel"],
                    output_channel=channel_config["output_channel"],
                    **kwargs
                )
                for channel_config in channels
            ]
        )

        self.pooling = nn.AdaptiveAvgPool2d(1)

        dummy_input = torch.ones(1, 1, 32, 32)
        with torch.no_grad():
            self.n_feat = self.conv_blocks(dummy_input).shape[1]

        self.classification = nn.Linear(in_features=self.n_feat, out_features=1)
        self.activate = nn.ReLU()
        self.decision = nn.Sigmoid()

    def forward(self, x):

        x = self.conv_blocks(x)
        x = self.pooling(x).reshape((-1, self.n_feat))
        x = self.classification(x)
        x = self.activate(x)
        return self.decision(x)


In [139]:
class ConvUpBlock(torch.nn.Module):

    def __init__(self, **kwargs):

        super().__init__()

        input_channel = kwargs.get("input_channel", 3)
        output_channel = kwargs.get("output_channel", 3)
        kernel_size = kwargs.get("kernel_size", 4)
        self.conv_transpose = nn.ConvTranspose2d(
            in_channels=input_channel,
            out_channels=output_channel,
            kernel_size=kernel_size,
            padding=1,
            stride=2,
        )
        # self.dropout = nn.Dropout()
        self.norm = nn.BatchNorm2d(output_channel)
        self.activate = nn.ReLU()

    def forward(self, x):

        x = self.conv_transpose(x)
        # x = self.dropout(x)
        x = self.norm(x)
        x = self.activate(x)
        return x


class Generator(nn.Module):

    def __init__(self, **kwargs):

        super().__init__()
        channels = kwargs.get("channels", [32, 16, 8, 4, 2, 1])

        channels = [
            {"input_channel": x, "output_channel": y}
            for x, y in zip(channels[:-1], channels[1:])
        ]
        self.conv_transpose_blocks = nn.Sequential(
            *[
                ConvUpBlock(
                    input_channel=channel_config["input_channel"],
                    output_channel=channel_config["output_channel"],
                    **kwargs
                )
                for channel_config in channels
            ]
        )

        self.decision = nn.Tanh()

    def forward(self, x):
        x = self.conv_transpose_blocks(x)
        return self.decision(x)


In [140]:
D = Discriminator()
G = Generator()
x = preprocess(train_images[0])
x.min(), x.max()

(tensor(-1.), tensor(0.9059))

In [141]:
x = torch.rand((1, 32, 1, 1))
y = G(x)
z = D(y)
z

tensor([[0.5505]], grad_fn=<SigmoidBackward0>)

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW


In [156]:
class MyDataset(Dataset):

    def __init__(self, folder, size_final=32):
        # folder = "/".join([os.path.expanduser("~/"), "Workspace/repos/fashion-mnist/data/fashion"])
        self.images, self.labels = load_mnist(folder)
        self.size_final=size_final

    def __len__(self):
        return len(self.images)
    
    def preprocess(self, x, ):

        n = int(math.sqrt(x.shape[0]))
        x = x.reshape((n, n, -1))

        x = cv2.resize(x, (self.size_final, self.size_final), interpolation=cv2.INTER_LINEAR)

        if len(x.shape) == 2:
            x = x[..., np.newaxis]
        x = np.transpose(x, (2, 0, 1))
    
        x = (x / MAX - 0.5) * 2

        return x

    def __getitem__(self, index):

        image = self.images[index]
        label = self.labels[index]

        image = self.preprocess(image)

        return {
            "image": image,
            "label": label
        }

In [ ]:
folder = "/".join([os.path.expanduser("~/"), "Workspace/repos/fashion-mnist/data/fashion"])
dataset = MyDataset(folder)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

D = Discriminator()
G = Generator()
D_optim = AdamW(D.parameters(), lr=1e-4)
G_optim = AdamW(G.parameters(), lr=1e-4)


In [ ]:
folder = "/".join([os.path.expanduser("~/"), "Workspace/repos/fashion-mnist/data/fashion"])
dataset = MyDataset(folder)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

D = Discriminator()
G = Generator()
D_optim = AdamW(D.parameters(), lr=1e-4)
G_optim = AdamW(G.parameters(), lr=1e-4)

D_loss = nn.BCELoss()
G_loss = nn.BCELoss()

for epoch in range(2):

    D_turn = True
    count = 0
    for batch in dataloader:
        real_images = batch["image"]

        real_pred = D(real_images.float())

        z = torch.rand((8, 32, 1, 1))
        generated_images = G(z)
        fake_pred = D(generated_images.float())

        if D_turn:
            count += 0
            D_target = torch.ones(16, dtype=torch.float32)
            D_target[8:] = 0
            pred = torch.concat([real_pred, 1 - fake_pred], dim=0).view(-1)
            loss = D_loss(pred, D_target)
            D.zero_grad()
            loss.backward()
            D_optim.step()
            if count > 3:
                D_turn = False
                count = 0
        else:
            G_target = torch.ones(8, dtype=torch.float32)
            loss = G_loss(fake_pred, G_target)
            G.zero_grad()
            loss.backward()
            G_optim.step()
            D_turn = True
        
        break
    break

    

In [185]:
loss

tensor(0.5244, grad_fn=<BinaryCrossEntropyBackward0>)